# Proyecto 2 - Análisis Exploratorio del dataset Diabetes 130-US

Este notebook cubre el punto **3.1** del enunciado: análisis del dataset `Diabetes.csv` antes de su limpieza, ingeniería de características y creación de subconjuntos de entrenamiento.

Las decisiones tomadas aquí justifican lo que después implementa el pipeline en `training/src/pipeline.py`:

- Tratamiento de nulos (`?` -> `NA`).
- Selección de features clínicos relevantes.
- Codificación del target `readmitted`.
- Estrategia de imputación + one-hot + scaling.
- Métrica principal de selección (`val_recall`).

## Cómo correrlo

```bash
cd "Proyecto 2"
python -m venv .venv && source .venv/bin/activate
pip install -r training/requirements.txt jupyter matplotlib seaborn
jupyter notebook notebooks/01_eda.ipynb
```

## 1. Carga del dataset

Reutilizamos `ensure_dataset_present()` del pipeline para tener una sola ruta de descarga. Esa función ya maneja el `confirm` token de Google Drive y valida que lo recibido sea un CSV real.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "training" / "src"))

os.environ.setdefault("DATASET_URL", "https://docs.google.com/uc?export=download&id=1k5-1caezQ3zWJbKaiMULTGq-3sz6uThC")
os.environ.setdefault("DATASET_PATH", str(PROJECT_ROOT / "data" / "Diabetes.csv"))

from pipeline import ensure_dataset_present

csv_path = ensure_dataset_present()
df = pd.read_csv(csv_path, na_values=["?"])
df.shape

In [ ]:
df.head()

In [ ]:
df.dtypes.value_counts()

## 2. Calidad de datos: nulos por columna

Reemplazamos los `?` por `NaN` (es el marcador de nulo del dataset original).

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0].to_frame("pct_missing").style.format({"pct_missing": "{:.2%}"})

**Observaciones esperadas** (al correr sobre el dataset modificado):

- `weight`, `medical_specialty` y `payer_code` suelen tener > 40% de nulos: candidatas a descartar.
- `race`, `diag_1`, `diag_2`, `diag_3` tienen pocos nulos (< 5%): se imputan.

Esto justifica el uso en el pipeline de:

- `SimpleImputer(strategy="median")` para numéricas.
- `SimpleImputer(strategy="most_frequent")` para categóricas.
- Selección manual de features clínicamente relevantes en `DEFAULT_FEATURE_COLUMNS`.

## 3. Distribución del target `readmitted`

Tres clases originales: `<30`, `>30`, `NO`. Para el problema de readmisión temprana lo más relevante clínicamente es **predecir `<30`** (readmisión en menos de 30 días) vs el resto. Por eso en el pipeline se binariza así.

In [ ]:
ax = df["readmitted"].value_counts(dropna=False).plot(kind="bar", color=["#4c72b0", "#55a868", "#c44e52"])
ax.set_title("Distribución original del target readmitted")
ax.set_ylabel("# encuentros")
plt.show()

df["readmitted"].value_counts(normalize=True).rename("share").to_frame()

In [ ]:
binary_target = df["readmitted"].eq("<30").astype(int)
positive_rate = binary_target.mean()
print(f"Tasa de positivos (readmisión < 30 días): {positive_rate:.2%}")

**Decisión:** el dataset tiene un fuerte desbalance hacia la clase negativa (~89% no readmiten en < 30 días). Por eso:

- No usamos `accuracy` como métrica principal (sería engañoso).
- Optamos por **`val_recall`** sobre la clase positiva: queremos detectar a los pacientes que sí van a ser readmitidos rápido (los falsos negativos son más costosos que los falsos positivos en este contexto clínico).
- También logueamos `precision`, `f1` y `roc_auc` en MLflow para tener trade-offs visibles.

## 4. Variables numéricas

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[numeric_cols].describe().T

In [ ]:
interesting_numeric = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, interesting_numeric):
    if col in df.columns:
        df[col].plot(kind="hist", bins=30, ax=ax, color="#4c72b0")
        ax.set_title(col)
for ax in axes.flat[len(interesting_numeric):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

**Observación:** las variables de conteo (`number_*`, `num_*`) están fuertemente sesgadas a la izquierda y con outliers. Esto motiva:

- Usar `StandardScaler` en lugar de `MinMaxScaler` (es robusto a sesgo cuando combinado con imputación por mediana).
- Considerar (a futuro) transformaciones tipo `log1p` o discretización de las variables de conteo extremas.

## 5. Variables categóricas y cardinalidad

In [ ]:
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
card = df[categorical_cols].nunique().sort_values(ascending=False)
card.to_frame("unique_values").head(20)

**Decisión:**

- Columnas como `diag_1/2/3` (códigos ICD-9) tienen cardinalidad altísima (cientos de valores). El `OneHotEncoder(handle_unknown="ignore")` las absorbe pero genera mucho ancho. Aceptable para LR/RF; si más adelante se quiere bajar dimensión, agrupar por categoría ICD.
- `encounter_id` y `patient_nbr` son identificadores -> se excluyen explícitamente del entrenamiento (`ID_COLUMNS` en el pipeline).

## 6. Relación target vs algunas categóricas

In [ ]:
df_plot = df.copy()
df_plot["target_lt_30"] = binary_target

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ["A1Cresult", "insulin", "diabetesMed"]):
    if col in df_plot.columns:
        rates = df_plot.groupby(col)["target_lt_30"].mean().sort_values()
        rates.plot(kind="bar", ax=ax, color="#c44e52")
        ax.set_title(f"Tasa de readmisión < 30 días por {col}")
        ax.set_ylabel("P(target=1)")
plt.tight_layout()
plt.show()

## 7. Correlaciones entre numéricas

In [ ]:
corr_cols = [c for c in interesting_numeric if c in df.columns]
corr = df[corr_cols + ["time_in_hospital"]].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", ax=ax)
ax.set_title("Correlación entre variables numéricas clínicas")
plt.show()

## 8. Conclusiones del EDA

1. El dataset tiene **alto desbalance** del target binario `<30` (~11% de positivos). Métrica principal: **`val_recall`**.
2. Hay columnas con > 40% de nulos (típicamente `weight`, `medical_specialty`, `payer_code`) que no se incluyen en `DEFAULT_FEATURE_COLUMNS`.
3. Las features clínicas seleccionadas son una mezcla de demográficas (`race`, `gender`, `age`), de uso hospitalario (`time_in_hospital`, `num_*`, `number_*`), de diagnóstico (`diag_1/2/3`) y de medicación (`A1Cresult`, `insulin`, `change`, `diabetesMed`).
4. Las variables numéricas tienen sesgo y outliers: imputación por mediana + `StandardScaler` es más estable que media+minmax.
5. Las categóricas se codifican con `OneHotEncoder(handle_unknown="ignore")` para tolerar valores nuevos en lotes futuros (clave porque la ingesta es **incremental** por batches).
6. `encounter_id` y `patient_nbr` son IDs y nunca entran al modelo.

Estas decisiones están materializadas en `training/src/pipeline.py` (`build_clean_layer`, `build_candidate_models`, `train_and_register_model`).